**Github Repository Link:**
https://github.com/s-foodie/uno-using-search

In [2]:
import random

class Card:
  colours = ["Red", "Yellow", "Blue", "Green"]
  numbers = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  special = ["Skip"]

  def __init__(self, colour, number):
    self.colour = colour
    self.number = number

  def show(self):
    return f"{self.colour} {self.number}"


def generatedeck():
  deck = []
  for colour in Card.colours:
    for number in Card.numbers:
      deck.append(Card(colour, number))
    deck.append(Card(colour, "Skip"))

  random.shuffle(deck)
  return deck


def dealcards(deck):
  p1 = []
  p2 = []
  p3 = []

  for i in range(5):
    p1.append(deck.pop())
    p2.append(deck.pop())
    p3.append(deck.pop())

  return p1, p2, p3


def showhand(hand):
  for i in range(len(hand)):
    print(i, ":", hand[i].show())


def validmoves(hand, topcard):
  moves = []

  for card in hand:
    if card.colour == topcard.colour:
      moves.append(card)

    elif card.number == topcard.number:
      moves.append(card)

    elif card.colour == topcard.colour and card.number == "Skip":
      moves.append(card)

  return moves


def evaluatedefensive(hand, opponent1, opponent2):
  cAI = len(hand)
  cOPPONENT = (len(opponent1) + len(opponent2))/2

  s = 0
  for card in hand:
    if card.number == "Skip":
      s = s + 1

  score = 50 - 5*cAI +2*cOPPONENT +3*s

  if len(opponent1) <= 2:
    score = score - 8

  if len(opponent2) <= 2:
    score = score - 8

  return score



def evaluateoffensive(hand, opponent1, opponent2):
  cAI = len(hand)
  cOPPONENT = (len(opponent1) + len(opponent2))/2

  s = 0
  for card in hand:
    if card.number == "Skip":
      s = s+1

  score = 50 - 7*cAI + 2*cOPPONENT + 2*s
  return score


def createstate(p1, p2, p3, topcard, deck, turn):
  return {"hands": [p1, p2, p3], "topcard":topcard, "deck":deck, "turn":turn}


def gameover(state):
  for i in range(3):
    if len(state["hands"][i]) == 0:
      return True
  return False

def applymove(state, move):
  newhands = [state["hands"][0].copy(), state["hands"][1].copy(), state["hands"][2].copy()]
  newdeck = state["deck"].copy()
  turn = state["turn"]
  hand = newhands[turn]
  topcard = state["topcard"]

  if move == "DRAW":
    if len(newdeck) > 0:
      hand.append(newdeck.pop())
    newturn = (turn + 1) % 3
    return {"hands": newhands, "topcard":topcard, "deck":newdeck, "turn":newturn}


  hand.remove(move)
  topcard = move
  if move.number == "Skip":
    newturn = (turn + 2) % 3

  else:
    newturn = (turn + 1) % 3

  return {"hands":newhands, "topcard":topcard, "deck":newdeck, "turn":newturn
          }


def minimaxvalue(state, depth, playerindex):
  if depth == 0 or gameover(state):
    hand = state["hands"][playerindex]
    opp = [state["hands"][i] for i in range(3) if i != playerindex]
    return evaluatedefensive(hand, opp[0], opp[1])


  turn = state["turn"]
  hand = state["hands"][turn]
  moves = validmoves(hand, state["topcard"])


  if len(moves) == 0:
    moves = ["DRAW"]

  if turn == playerindex:
    best = -99999
    for move in moves:
      child = applymove(state, move)
      value = minimaxvalue(child, depth - 1, playerindex)
      if value > best:
        best = value

    return best

  else:
    best = 99999
    for move in moves:
      child = applymove(state, move)
      value = minimaxvalue(child, depth - 1, playerindex)
      if value < best:
        best = value
    return best


def minimax(state, playerindex, depth):
  hand = state["hands"][playerindex]
  moves = validmoves(hand, state["topcard"])

  if len(moves) == 0:
    return "DRAW"

  bestmove = None
  bestscore = -99999

  print("AI decision: ")

  for move in moves:
    child = applymove(state, move)
    score = minimaxvalue(child, depth - 1, playerindex)
    print("Play:", move.show(), "Expected score:", score)

    if score > bestscore:
      bestscore = score
      bestmove = move

  return bestmove


def expectimaxvalue(state, depth, playerindex):
  if depth == 0 or gameover(state):
    hand = state["hands"][playerindex]
    opp = [state["hands"][i] for i in range(3) if i != playerindex]

    return evaluateoffensive(hand, opp[0], opp[1])

  turn = state["turn"]
  hand = state["hands"][turn]
  moves = validmoves(hand, state["topcard"])

  if turn == playerindex:
    if len(moves) == 0:
      if len(state["deck"]) == 0:
        child = applymove(state, "DRAW")
        return expectimaxvalue(child, depth - 1, playerindex)

      total = 0
      for i in range(len(state["deck"])):
        child = {"hands":[state["hands"][0].copy(), state["hands"][1].copy(), state["hands"][2].copy()], "deck": state["deck"].copy(), "topcard":state["topcard"], "turn": state["turn"]}
        drawn = child["deck"].pop(i)
        child["hands"][turn].append(drawn)
        child["turn"] = (turn + 1) % 3
        prob = 1 / len(state["deck"])
        total = total + prob * expectimaxvalue(child, depth - 1, playerindex)
      return total

    best = -99999
    for move in moves:
      child = applymove(state, move)
      value = expectimaxvalue(child, depth - 1, playerindex)
      if value > best:
        best = value
    return best

  else:
    if len(moves) == 0:
      child = applymove(state, "DRAW")
      return expectimaxvalue(child, depth - 1, playerindex)

    total = 0
    prob = 1 / len(moves)

    for move in moves:
      child = applymove(state, move)
      total = total + prob*expectimaxvalue(child, depth - 1, playerindex)

    return total



def expectimax(state, playerindex, depth):
  hand = state["hands"][playerindex]
  moves = validmoves(hand, state["topcard"])

  if len(moves) == 0:
    return "DRAW"

  bestscore = -99999
  bestmove = None

  for move in moves:
    child = applymove(state, move)
    score = expectimaxvalue(child, depth -1, playerindex)
    print("Play:", move.show(), "Expected Score:", round(score, 2))

    if score > bestscore:
      bestscore = score
      bestmove = move

  return bestmove



def player3(state):
  hand = state["hands"][2]
  moves = validmoves(hand, state["topcard"])

  print("Your hand: ")
  showhand(hand)

  validindexes = []

  if len(moves) == 0:
    print("No valid moves. You must draw")
    return "DRAW"

  print("Valid moves are:")
  for i in range(len(hand)):
    if hand[i] in moves:
      print(i, ":", hand[i].show())
      validindexes.append(i)

  while True:
    choice = int(input("Enter card index to play: "))
    if choice in validindexes:
      return hand[choice]
    else:
      print("Invalid choice. Choose from", validindexes)


def player3simulation(state, depth):
  return minimax(state, 2, depth)


def game():
  deck = generatedeck()
  p1, p2, p3 = dealcards(deck)

  topcard = deck.pop()

  while topcard.number == "Skip":
    deck.insert(0, topcard)
    random.shuffle(deck)
    topcard = deck.pop()


  print("Choose Player 3 mode:")
  print("1. Manual")
  print("2. Simulation")
  mode = input("Enter mode: ")

  state = createstate(p1, p2, p3, topcard, deck, 0)
  turn = 0

  while True:
    print("Top card: ", state["topcard"].show())
    print("Player 1 cards:", len(state["hands"][0]))
    print("Player 2 cards:", len(state["hands"][1]))
    print("Player 3 cards:", len(state["hands"][2]))


    if gameover(state):
      break

    turn = state["turn"]

    if turn == 0:
      print("Player 1's turn (using Minimax)")
      print ("Hand:")
      showhand(state["hands"][0])
      move = minimax(state, 0, 3)

    elif turn == 1:
      print("Player 2's turn (using Expectimax)")
      print("Hand: ")
      showhand(state["hands"][1])
      move = expectimax(state, 1, 3)

    else:
      if mode == "1":
        print("Player 3's turn (manual)")
        move = player3(state)

      else:
        print("Player 3's turn (simulation)")
        print("Hand:")
        showhand(state["hands"][2])
        move = player3simulation(state, 3)

    state = applymove(state, move)

    if move == "DRAW":
      print("Player", ((state["turn"] - 1) % 3) + 1, "draws a card")
    else:
      print("Player played:", move.show())
  print("Game Over")
  for i in range(3):
    if len(state["hands"][i]) == 0:
      print("Player", i + 1, "wins")


game()




Choose Player 3 mode:
1. Manual
2. Simulation
Enter mode: 1
Top card:  Yellow 8
Player 1 cards: 5
Player 2 cards: 5
Player 3 cards: 5
Player 1's turn (using Minimax)
Hand:
0 : Yellow 0
1 : Blue 0
2 : Blue 2
3 : Yellow 2
4 : Blue 1
AI decision: 
Play: Yellow 0 Expected score: 40.0
Play: Yellow 2 Expected score: 40.0
Player played: Yellow 0
Top card:  Yellow 0
Player 1 cards: 4
Player 2 cards: 5
Player 3 cards: 5
Player 2's turn (using Expectimax)
Hand: 
0 : Blue Skip
1 : Green 3
2 : Green 2
3 : Blue 8
4 : Green 5
Player 2 draws a card
Top card:  Yellow 0
Player 1 cards: 4
Player 2 cards: 6
Player 3 cards: 5
Player 3's turn (manual)
Your hand: 
0 : Yellow 7
1 : Yellow 6
2 : Yellow 5
3 : Red 9
4 : Yellow 3
Valid moves are:
0 : Yellow 7
1 : Yellow 6
2 : Yellow 5
4 : Yellow 3
Enter card index to play: 0
Player played: Yellow 7
Top card:  Yellow 7
Player 1 cards: 4
Player 2 cards: 6
Player 3 cards: 4
Player 1's turn (using Minimax)
Hand:
0 : Blue 0
1 : Blue 2
2 : Yellow 2
3 : Blue 1
AI decis

# **Explanation of Evaluation Function**

The evaluation function helps the AI decide which move is better and how good the current game situation is. It helps AI choose moves that would increase its chances of winning the game.


Here, the evaluation function is:

Score = 50 - 5(CAI) + 2(Copp) + 3(S)

The AI wants to have fewer cards because the goal of UNO is to finish all cards first. Therefore, when the AI has fewer cards, the score becomes higher. If the opponents have more cards, the score increases then too. Skip cards allow the AI to skip the next player's turn, so these increase the score as well.

Minimax used by Player 1 uses this evaluation function to play defensively and prevent opponents from winning.

Expectimax used by Player 2 uses this evaluation function but also considers uncertainity like drawing cards the the deck, and chooses the move with the best expected result.




# **Comparision of Minimax and Expectimax**

Minimax, used by Player 1, performed better in situations where opponents were close to winning because it assumes opponents will choose moves that are worst for the AI. This makes it more defensive.


Expectimax performed better in situations where aggressive play was more useful because it evaluates the average outcome of uncertain events such as drawing cards from the deck. This makes it more offensive.


The better algorithm depends on the situation: Minimax was more stronger defensively and Expectimax was stronger offensively.

# **Generated Search Tree**


**Player 1**

Blue 5 -> score 38

Blue 0 -> score 38

Best Move: Blue 5


**Player 2**

Green 5 -> score 33

Best Move: Green 5


**Player 3**

Red 5 -> score 36

Green 9 -> score 36

Best Move: Red 5


**Player 1**

Red 3 -> score 37

Best Move: Red 3


**Player 2**

Red 0 -> score 40

Red Skip -> score 46

Best Move: Red Skip


**Player 3**

Turn Skipped


**Player 1**

Draws Card


**Player 2**

Red 0 -> score 46

Green Skip -> score 50

Best Move: Green Skip


**Player 1**

Green 8 -> score 31

Green 3 -> score 31

Best Move: Green 8


**Player 2**

Green 1 -> score 48

Best Move: Green 1


**Player 3**

Green 9 -> score 23

Best Move: Green 9


**Player 1**

Green 3 -> score 38

Best Move: Green 3


**Player 2**

Draws Card


**Player 3**

Draws Card


**Player 1**

Draws Card


**Player 2**

Draws Card


**Player 3**

Green 4 -> score 33

Best Move: Green 4


**Player 1**

Draws Card


**Player 2**

Red 4 -> score 41

Best Move: Red 4


**Player 3**

Red 7 -> score 46

Best Move: Red 7


**Player 1**

Yellow 7 -> score 31

Best Move: Yellow 7


**Player 2**

Draws card


**Player 3**

Yellow 6 -> score 33

Best Move: Yellow 6


**Player 1**

Yellow 1 -> score 38

Yellow 3 -> score 26

Best Move: Yellow 1


**Player 2**

Draws Card


**Player 3**

Draws Card


**Player 1**

Yellow 3 -> score 41

Best Move: Yellow 3


**Player 2**

Blue 3 -> score 30

Best Move: Blue 3


**Player 3**

Blue 6 -> score 40

Best Move: Blue 6


**Player 1**

Blue 0 -> score 46

Best Move: Blue 0


**Player 1 wins**




